# Chapter 6 & 7 — Fine-Tuning

**Goal:** Adapt the pre-trained model for specific tasks.

### Two approaches covered
1. **Classification** — attach a linear head, freeze base model, train on labeled data
2. **Instruction following** — continue training the full model on (instruction, response) pairs

In [ ]:
import sys, torch
sys.path.insert(0, '..')
from src.model.gpt import GPT, GPTConfig
from src.training.finetune import attach_classifier, InstructionDataset, finetune, collate_fn
from torch.utils.data import DataLoader

## Part A — Classification Fine-Tuning (e.g. spam detection)

In [ ]:
cfg   = GPTConfig(vocab_size=50257, context_length=32, embed_dim=128, num_heads=4, num_layers=2)
model = GPT(cfg)

# Optionally load pre-trained weights:
# model.load_state_dict(torch.load('../data/gpt_pretrained.pt', map_location='cpu'))

model = attach_classifier(model, num_classes=2, freeze_base=True)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable:,}')   # only the classifier head

## Part B — Instruction Fine-Tuning

In [ ]:
# Sample instruction data (replace with real dataset e.g. Alpaca)
sample_data = [
    {
        'instruction': 'Explain what a neural network is.',
        'input': '',
        'output': 'A neural network is a series of algorithms that attempt to recognize patterns in data.'
    },
    {
        'instruction': 'Translate to French.',
        'input': 'Hello, how are you?',
        'output': 'Bonjour, comment allez-vous?'
    },
]

from src.data.tokenizer import BPETokenizer
tokenizer = BPETokenizer()
ds = InstructionDataset(sample_data, tokenizer, context_length=64)
print(f'Instruction dataset: {len(ds)} samples')

loader = DataLoader(ds, batch_size=2, collate_fn=collate_fn)
x, y = next(iter(loader))
print(f'Batch input shape:  {x.shape}')
print(f'Batch target shape: {y.shape}')

## Exercises
1. Load a real spam/ham dataset (e.g. SMS Spam Collection) and fine-tune the classifier.
2. What is the minimum number of instruction examples needed to see meaningful behavior change?
3. How does `freeze_base=True` vs `freeze_base=False` affect training speed and accuracy?